# 5. Automated ETL pipeline

Run `python -m pip install -e ".[dev]"` from the repository root first. Each notebook runs independently and imports the same tested Python functions. Source timestamps retain their original local calendar meaning. Set `RETAIL_ETL_INPUT` to use another source and adjust `AS_OF` to its reporting cutoff.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from retail_etl.pipeline import project_root, read_source
from retail_etl.transform import prepare_data, feature_engineering, validate_output

ROOT = project_root()
SOURCE = Path(os.getenv("RETAIL_ETL_INPUT", str(ROOT / "data/raw/Online Retail.xlsx")))
OUTPUT = Path(os.getenv("RETAIL_ETL_OUTPUT", str(ROOT / "outputs")))
AS_OF = "2011-12-09"  # Inclusive end date of the bundled historical dataset.


## Run extraction through verified export

The shared pipeline creates an immutable run directory with cleaned data, final data, excluded records, validation, a manifest, and a rotating log. The latest pointer advances only after success. Exceptions are logged and re-raised.

In [ ]:
from retail_etl.pipeline import run_pipeline
report, final = run_pipeline(SOURCE, output_root=OUTPUT, as_of=AS_OF)
report

## Review publication checks

In [ ]:
print("Run:", report["run_id"])
print("Checks:", report["output_checks"])
print("Excluded:", report["validation"]["rejected_rows"])
final.head()

## Optional MySQL publication

Configure the documented MYSQL_* environment variables and run `python -m retail_etl --as-of 2011-12-09 --load-mysql` from a terminal. The publisher checks a staging table before an atomic rename and retains the previous table as a backup. Running this notebook never changes your database by default. See `docs/database.md` for migration and restore details.